# Notebook 05 — Microbiome × Villus Layer Interaction

**Scientific question:** Does the microbiome (SPF vs GF) affect gene 
expression differently depending on villus layer position?

Mayassi et al. 2024 showed overall transcriptional robustness to 
microbiome presence across gut regions. We extend this finding by 
asking whether robustness is uniform across the crypt-to-villus axis, 
or whether specific layers are more microbiome-sensitive than others.

**Hypothesis:** The villus tip, being directly exposed to the gut lumen, 
is more transcriptionally responsive to microbiome presence than the 
crypt, which is physically protected from luminal bacteria.

**Design:**
- Load SI epithelial subset (saved from notebook 03)
- Compare SPF vs GF within each region × layer combination
- Downsample to equal spot counts for fair comparison
- Count DEGs per combination and visualize
- GO enrichment on the most microbiome-sensitive layer

## 1. Imports and Paths

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp
import psutil
import warnings
import os
import gc
warnings.filterwarnings('ignore')

sc.settings.verbosity = 1

PROCESSED_DIR = "/home/wojmor/studies/MODEL/gut_project/data/processed"
FIGURES_DIR  = "/home/wojmor/studies/MODEL/gut_project/results/figures"
TABLES_DIR   = "/home/wojmor/studies/MODEL/gut_project/results/tables"

os.makedirs(TABLES_DIR, exist_ok=True)
print("Imports done.")

Imports done.


## 2. Load SI Epithelial Subset

We load the pre-subsetted AnnData saved at the end of notebook 03 — 
small intestine epithelial spots only (Crypt SI, Bottom villous SI, 
Top villous SI across all four SI regions). At 5.6GB this is 
significantly smaller than the full 8GB atlas.

In [ ]:
adata_epi = sc.read_h5ad(f"{PROCESSED_DIR}/visium_si_epithelial.h5ad")

mem = psutil.virtual_memory()
print(f"Loaded: {adata_epi.n_obs} spots × {adata_epi.n_vars} genes")
print(f"Memory used: {mem.used / 1e9:.1f} GB / available: {mem.available / 1e9:.1f} GB")
print(f"\nSpots per condition:")
print(adata_epi.obs['microbe'].value_counts())
print(f"\nSpots per layer:")
print(adata_epi.obs['ylayer'].value_counts())
print(f"\nSpots per region:")
print(adata_epi.obs['tissue_region'].value_counts())

## 3. SPF vs GF DEG Analysis With Downsampling

For each of the 12 region × layer combinations we:
1. Subset to SPF and GF spots only
2. Downsample the larger condition to match the smaller one
3. Run Wilcoxon rank-sum test (SPF vs GF)
4. Extract DEGs in both directions

Downsampling ensures equal statistical power for both conditions, 
making DEG counts directly comparable across combinations.
Reproducibility is ensured with a fixed random seed (42).

In [ ]:
regions = ['Duodenum', 'Jejunum1', 'Jejunum2', 'Ileum']
layers  = ['Crypt SI', 'Bottom villous SI', 'Top villous SI']

spf_gf_results = {}
deg_counts     = []

for region in regions:
    for layer in layers:
        # Subset to this region × layer, SPF and GF only
        mask = (
            (adata_epi.obs['tissue_region'] == region) &
            (adata_epi.obs['ylayer'] == layer) &
            (adata_epi.obs['microbe'].isin(['SPF', 'GF']))
        )
        adata_sub = adata_epi[mask].copy()

        spf_idx = adata_sub.obs_names[adata_sub.obs['microbe'] == 'SPF']
        gf_idx  = adata_sub.obs_names[adata_sub.obs['microbe'] == 'GF']
        n_spf, n_gf = len(spf_idx), len(gf_idx)

        if n_spf < 50 or n_gf < 50:
            print(f"Skipping {region} × {layer}: SPF={n_spf}, GF={n_gf}")
            del adata_sub; gc.collect()
            continue

        # Downsample to smaller condition
        n_sample = min(n_spf, n_gf)
        np.random.seed(42)
        spf_sampled = np.random.choice(spf_idx, n_sample, replace=False)
        gf_sampled  = np.random.choice(gf_idx,  n_sample, replace=False)
        adata_ds = adata_sub[list(spf_sampled) + list(gf_sampled)].copy()
        del adata_sub; gc.collect()

        # DEG analysis
        adata_ds.obs['microbe'] = adata_ds.obs['microbe'].astype('category')
        sc.tl.rank_genes_groups(
            adata_ds,
            groupby='microbe',
            groups=['SPF'],
            reference='GF',
            method='wilcoxon',
            key_added='degs_spf_vs_gf',
            pts=True
        )

        # Extract results in both directions
        df = sc.get.rank_genes_groups_df(
            adata_ds,
            group='SPF',
            key='degs_spf_vs_gf',
            pval_cutoff=0.05,
        )
        df = df[np.isfinite(df['logfoldchanges'])].copy()
        df['region'] = region
        df['layer']  = layer
        del adata_ds; gc.collect()

        n_spf_up = (df['logfoldchanges'] > 0).sum()
        n_gf_up  = (df['logfoldchanges'] < 0).sum()

        spf_gf_results[f"{region}_{layer}"] = df
        deg_counts.append({
            'region':    region,
            'layer':     layer,
            'n_spf_up':  n_spf_up,
            'n_gf_up':   n_gf_up,
            'n_total':   len(df),
            'n_sample':  n_sample
        })

        print(f"{region} × {layer}: "
              f"SPF↑={n_spf_up}, GF↑={n_gf_up} "
              f"(n={n_sample} per condition)")

# Save full DEG results
all_results = pd.concat(spf_gf_results.values(), ignore_index=True)
all_results.to_csv(f"{TABLES_DIR}/spf_vs_gf_bidirectional.csv", index=False)

deg_counts_df = pd.DataFrame(deg_counts)
deg_counts_df.to_csv(f"{TABLES_DIR}/spf_vs_gf_deg_counts.csv", index=False)

print(f"\nAll results saved.")

mem = psutil.virtual_memory()
print(f"Memory used: {mem.used / 1e9:.1f} GB / available: {mem.available / 1e9:.1f} GB")

## 4. Figure — Microbiome Sensitivity Heatmap

We visualize SPF vs GF DEG counts per region × layer combination. 
Separate panels show genes promoted by the microbiome (SPF↑) 
and genes suppressed by the microbiome (GF↑).

In [ ]:
layer_order  = ['Crypt SI', 'Bottom villous SI', 'Top villous SI']
region_order = ['Duodenum', 'Jejunum1', 'Jejunum2', 'Ileum']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax_idx, (col, title, cmap) in enumerate([
    ('n_spf_up', 'SPF-upregulated\n(microbiome promotes)', 'YlOrRd'),
    ('n_gf_up',  'GF-upregulated\n(microbiome suppresses)', 'YlGnBu')
]):
    pivot = deg_counts_df.pivot(
        index='layer', columns='region', values=col
    ).reindex(index=layer_order, columns=region_order)

    sns.heatmap(
        pivot, ax=axes[ax_idx],
        cmap=cmap, annot=True, fmt='.0f',
        linewidths=0.5,
        cbar_kws={'label': 'Number of DEGs'}
    )
    axes[ax_idx].set_title(title, fontweight='bold', fontsize=11)
    axes[ax_idx].set_xlabel('SI Region')
    axes[ax_idx].set_ylabel('Villus Layer')

plt.suptitle(
    'Microbiome sensitivity across villus layers and SI regions\n'
    '(SPF vs GF, downsampled to equal spot counts)',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/05_microbiome_sensitivity_heatmap.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")

## 5. GO Enrichment on Most Microbiome-Sensitive Layer

The bottom villous SI is consistently the most microbiome-sensitive 
layer across all SI regions. We perform GO enrichment to identify 
which biological processes the microbiome promotes or suppresses 
in this layer.

In [ ]:
# Pool bottom villous DEGs across all four regions
bottom_dfs = [
    spf_gf_results[f"{r}_Bottom villous SI"]
    for r in regions
    if f"{r}_Bottom villous SI" in spf_gf_results
]
bottom_degs = pd.concat(bottom_dfs, ignore_index=True)

# Split by direction with meaningful fold change threshold
spf_up = bottom_degs[bottom_degs['logfoldchanges'] >  0.25]['names'].unique().tolist()
gf_up  = bottom_degs[bottom_degs['logfoldchanges'] < -0.25]['names'].unique().tolist()

print(f"Bottom villous SI — SPF-upregulated: {len(spf_up)} genes")
print(f"Bottom villous SI — GF-upregulated:  {len(gf_up)} genes")

In [ ]:
go_results = {}

for direction, gene_list, color in [
    ('SPF-upregulated', spf_up, '#e41a1c'),
    ('GF-upregulated',  gf_up,  '#377eb8')
]:
    if len(gene_list) < 10:
        print(f"Skipping {direction} — too few genes ({len(gene_list)})")
        continue

    print(f"Running GO: {direction} ({len(gene_list)} genes)...")
    res = gp.enrichr(
        gene_list=gene_list,
        gene_sets=['GO_Biological_Process_2023'],
        organism='mouse',
        outdir=None,
        cutoff=0.05
    ).results

    sig = res[res['Adjusted P-value'] < 0.05].sort_values('Adjusted P-value')
    go_results[direction] = (sig, color)
    sig.to_csv(
        f"{TABLES_DIR}/go_bottom_villous_{direction.lower().replace('-','_')}.csv",
        index=False
    )
    print(f"  → {len(sig)} significant terms saved.")

## 6. Figure — GO Enrichment in Bottom Villous SI

In [ ]:
def plot_go_terms(df, title, color, ax, n_terms=12):
    bp = df[df['Gene_set'].str.contains('Biological_Process')].copy()
    if len(bp) == 0:
        ax.text(0.5, 0.5, 'No significant terms',
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title, fontweight='bold')
        return
    top = bp.nsmallest(n_terms, 'Adjusted P-value').copy()
    top['Term_clean'] = top['Term'].str.replace(
        r'\(GO:\d+\)', '', regex=True).str.strip()
    top['neg_log_p'] = -np.log10(top['Adjusted P-value'])
    ax.barh(range(len(top)), top['neg_log_p'].values,
            color=color, alpha=0.8, edgecolor='black', linewidth=0.3)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(
        [t[:55] + '...' if len(t) > 55 else t
         for t in top['Term_clean'].values], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('-log10(adjusted p-value)')
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.axvline(x=-np.log10(0.05), color='red',
               linestyle='--', alpha=0.5, label='p=0.05')

n_panels = len(go_results)
fig, axes = plt.subplots(1, n_panels, figsize=(12 * n_panels, 7))
if n_panels == 1:
    axes = [axes]

for ax_idx, (direction, (sig, color)) in enumerate(go_results.items()):
    plot_go_terms(
        sig,
        f'Bottom Villous SI\n{direction}\n(GO Biological Process)',
        color, axes[ax_idx]
    )

plt.suptitle(
    'GO enrichment — microbiome-sensitive genes in Bottom Villous SI',
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/05_go_bottom_villous.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")